In [19]:
from lightgbm import LGBMClassifier
import pandas as pd 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

df = pd.read_csv("../datasets/reddit.csv")
df.rename(columns={'clean_text': 'message', 'is_depression': 'label'}, inplace=True)
X_train, X_test, y_train, y_test = train_test_split(df['message'], df['label'], test_size=0.3, random_state=42)
lgbm_model = LGBMClassifier(n_jobs=-1, reg_lambda=0.1, num_leaves=64, max_depth=-1, learning_rate=0.1, n_estimators=100)
lgbm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000, 
        ngram_range=( 1,3), 
        stop_words='english', 
        lowercase=True, 
        norm='l2', 
        use_idf=True, 
        smooth_idf=True, 
        sublinear_tf=True
    )),
    ('classifier', lgbm_model)
])
lgbm_pipeline.fit(X_train, y_train)
y_pred_lgbm = lgbm_pipeline.predict(X_test)
classification_report_result = classification_report(y_test, y_pred_lgbm)
print(classification_report_result)

[LightGBM] [Info] Number of positive: 2688, number of negative: 2723
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.077443 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 36105
[LightGBM] [Info] Number of data points in the train set: 5411, number of used features: 1390
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.496766 -> initscore=-0.012937
[LightGBM] [Info] Start training from score -0.012937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1177
           1       0.97      0.93      0.95      1143

    accuracy                           0.95      2320
   macro avg       0.95      0.95      0.95      2320
weighted 

/home/hyo/Documents/projects/trackerv2/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [26]:
joblib.dump(lgbm_pipeline, 'lightgbm.pkl')

['lightgbm.pkl']

In [27]:
message = "what a beautiful day, I feel so happy and grateful for everything"
predicted_label = lgbm_pipeline.predict([message])
print(f"Predicted label for the message: {predicted_label[0]}")

Predicted label for the message: 0


/home/hyo/Documents/projects/trackerv2/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
